# Trabalho Prático 2 - Inteligência Artificial - UFRRJ
Aluno: Pedro Costa da Motta

Matrícula: 20240014240

### Escolha da base de dados

Neste trabalho foi utilizado o conjunto de dados Iris, disponível na biblioteca Scikit-Learn.  O dataset contém exemplos de três espécies de flores (Setosa, Versicolor e Virginica) e quatro características numéricas relacionadas às dimensões das sépalas e pétalas.
A escolha desse conjunto de dados foi motivada pela sua simplicidade e pela facilidade de interpretação das regras geradas pela árvore de decisão.

A base de dados possui apenas atributos numéricos, mas a árvore de decisão consegue transformá-los em regras facilmente interpretáveis através dos limiares de decisão, o que seria análogo a uma categorização. Na prática, os testes realizados pela árvore dividem as variáveis contínuas em intervalos. Dessa forma, foi possível traduzir as regras para Prolog sem necessidade de realizar uma discretização manual das características.

In [ ]:
# 1. Importar as bibliotecas necessárias
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 2. Carregar o dataset Iris
iris = load_iris()
X = iris.data
y = iris.target

###  Treinamento da Árvore de Decisão

Os dados foram divididos utilizando o regime Holdout, reservando 80% das amostras para treinamento e 20% para teste. Em seguida foi treinada uma árvore de decisão com profundidade máxima igual a 3, de forma a manter a árvore simples e facilitar sua tradução para Prolog.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
plt.figure(figsize=(12, 8))
plot_tree(clf, feature_names=iris.feature_names, class_names=iris.target_names, filled=True)
plt.show()

### Avaliação da Árvore de Decisão

A matriz de confusão mostrou que todas as amostras do conjunto de teste foram classificadas corretamente. Não houve erros de classificação entre as espécies avaliadas.

Como consequência, os valores de Precisão, Recall e F1-Score foram iguais a 1,0 para todas as classes. Isso indica que o modelo conseguiu identificar corretamente todas as amostras presentes no conjunto de teste, sem produzir falsos positivos ou falsos negativos.

In [ ]:
y_pred = clf.predict(X_test)
print("=== MATRIZ DE CONFUSÃO ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print("\n" + "="*30 + "")

n_classes = len(iris.target_names)
for i in range(n_classes):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )
    print(f"\nClasse: {iris.target_names[i]}")
    print(f"Precisão = {precision:.4f}")
    print(f"Recall    = {recall:.4f}")
    print(f"F1-score  = {f1:.4f}")

### Discussão dos Resultados

A árvore gerada apresentou uma estrutura simples e de fácil interpretação. A primeira divisão da árvore utiliza o comprimento da pétala e já é suficiente para separar completamente a espécie Setosa das demais. Isso mostra que essa espécie possui características bastante distintas dentro do conjunto de dados.

Para diferenciar Versicolor e Virginica, a árvore passou a utilizar combinações entre comprimento e largura das pétalas. Observa-se que apenas características relacionadas às pétalas foram utilizadas nas decisões finais, enquanto as características das sépalas não apareceram nas regras aprendidas. Isso sugere que as medidas das pétalas possuem maior relevância para a classificação das espécies.

Os resultados obtidos indicam que o problema de classificação não apresenta elevada dificuldade para esse conjunto de dados, já que uma árvore relativamente pequena foi capaz de atingir desempenho perfeito no conjunto de teste.

Também não foram observados indícios de underfitting, pois o modelo conseguiu representar adequadamente os padrões presentes nos dados. Da mesma forma, não há evidências claras de overfitting, uma vez que a árvore possui profundidade limitada e manteve uma estrutura simples, sem a necessidade de criar um grande número de regras para realizar a classificação.

### Base de Conhecimento Prolog

```prolog
classificar(PL, PW, 'setosa') :-
    PL <= 2.45.

classificar(PL, PW, 'versicolor') :-
    PL > 2.45,
    PL <= 4.75,
    PW <= 1.65.

classificar(PL, PW, 'virginica') :-
    PL > 2.45,
    PL <= 4.75,
    PW > 1.65.

classificar(PL, PW, 'versicolor') :-
    PL > 4.75,
    PW <= 1.75.

classificar(PL, PW, 'virginica') :-
    PL > 4.75,
    PW > 1.75.
```

### Testes

Para verificar se a tradução da árvore de decisão para Prolog foi realizada corretamente, foram selecionados cinco exemplos do conjunto de teste. A classificação desses exemplos foi obtida através da função predict() da biblioteca Scikit-Learn.
A função predict() recebe como entrada um exemplo contendo as características da flor e percorre a árvore de decisão treinada até alcançar uma folha. A classe associada a essa folha é então retornada como resultado da classificação.

Também há uma tabela com o resultado em Prolog (feito manualmente)

In [ ]:
for i in range(5):
    print(f"\nExemplo {i+1}")
    print("Características:")
    print(X_test[i])
    predicao = clf.predict([X_test[i]])[0]
    print("Classe prevista pela árvore:")
    print(iris.target_names[predicao])

| Exemplo | Comprimento da Pétala (PL) | Largura da Pétala (PW) | Classe prevista pela árvore | Regra Prolog utilizada | Classe obtida |
| ------- | -------------------------- | ---------------------- | --------------------------- | ---------------------- | ------------- |
| 1       | 4.7                        | 1.2                    | Versicolor                  | Regra 2                | Versicolor    |
| 2       | 1.7                        | 0.3                    | Setosa                      | Regra 1                | Setosa        |
| 3       | 6.9                        | 2.3                    | Virginica                   | Regra 5                | Virginica     |
| 4       | 4.5                        | 1.5                    | Versicolor                  | Regra 2                | Versicolor    |
| 5       | 4.8                        | 1.4                    | Versicolor                  | Regra 4                | Versicolor    |
